In [1]:
# Step 1: Import Libraries
import pandas as pd
import boto3
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Step 2: Load Data
bucket = "flight-delay-mlops-project"
s3 = boto3.client('s3')

s3.download_file(bucket, 'flights.csv', 'flights.csv')

df = pd.read_csv('flights.csv', nrows=50000)

df.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,408.0,-22.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,741.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,811.0,5.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,756.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,259.0,-21.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
# Step 3: Data Cleaning
# Removing unnecessary columns, handling missing values, and removing duplicates

df = df.drop(['CANCELLATION_REASON'], axis=1)
df = df.fillna(0)
df = df.drop_duplicates()

df.isnull().sum()

YEAR                   0
MONTH                  0
DAY                    0
DAY_OF_WEEK            0
AIRLINE                0
FLIGHT_NUMBER          0
TAIL_NUMBER            0
ORIGIN_AIRPORT         0
DESTINATION_AIRPORT    0
SCHEDULED_DEPARTURE    0
DEPARTURE_TIME         0
DEPARTURE_DELAY        0
TAXI_OUT               0
WHEELS_OFF             0
SCHEDULED_TIME         0
ELAPSED_TIME           0
AIR_TIME               0
DISTANCE               0
WHEELS_ON              0
TAXI_IN                0
SCHEDULED_ARRIVAL      0
ARRIVAL_TIME           0
ARRIVAL_DELAY          0
DIVERTED               0
CANCELLED              0
AIR_SYSTEM_DELAY       0
SECURITY_DELAY         0
AIRLINE_DELAY          0
LATE_AIRCRAFT_DELAY    0
WEATHER_DELAY          0
dtype: int64

In [3]:
# Step 4: Feature Engineering
# Creating target variable (DELAYED) and removing leakage-causing features
# IMPORTANT: Removed ARRIVAL_DELAY to avoid data leakage

df = df[['AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT',
         'DEPARTURE_DELAY', 'DISTANCE', 'DAY_OF_WEEK', 'ARRIVAL_DELAY']]

df['DELAYED'] = df['ARRIVAL_DELAY'].apply(lambda x: 1 if x > 0 else 0)

# Check class balance
print("\nClass Distribution:\n")
print(df['DELAYED'].value_counts())

# Drop ARRIVAL_DELAY after creating target
df = df.drop(['ARRIVAL_DELAY'], axis=1)

df.head()


Class Distribution:

DELAYED
1    26053
0    23947
Name: count, dtype: int64


,AIRLINE,ORIGIN_AIRPORT,DESTINATION_AIRPORT,DEPARTURE_DELAY,DISTANCE,DAY_OF_WEEK,DELAYED
0,AS,ANC,SEA,-11.0,1448,4,0
1,AA,LAX,PBI,-8.0,2330,4,0
2,US,SFO,CLT,-2.0,2296,4,1
3,AA,LAX,MIA,-5.0,2342,4,0
4,AS,SEA,ANC,-1.0,1448,4,0


In [4]:
# Step 5: Feature Transformation
# Converting categorical variables into numerical format using one-hot encoding

df = pd.get_dummies(df, drop_first=True)

# Ensure no categorical (object) columns remain
print("Remaining object columns:", df.select_dtypes(include='object').columns)

Remaining object columns: Index([], dtype='object')


In [5]:
# Step 6: Train-Test Split
# Splitting dataset into training and testing sets (80/20)

X = df.drop(['DELAYED'], axis=1)
y = df['DELAYED']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
# Step 7: Model Training
# Training Random Forest model to classify delayed vs non-delayed flights

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Feature importance (understanding model behavior)
import pandas as pd

feature_importance = pd.Series(model.feature_importances_, index=X.columns)
feature_importance = feature_importance.sort_values(ascending=False)

print("\nTop Important Features:\n")
print(feature_importance.head(10))


Top Important Features:

DEPARTURE_DELAY            0.498388
DAY_OF_WEEK                0.070005
DISTANCE                   0.068545
AIRLINE_DL                 0.008915
DESTINATION_AIRPORT_IAH    0.006898
AIRLINE_WN                 0.005419
ORIGIN_AIRPORT_DEN         0.005291
AIRLINE_UA                 0.005258
ORIGIN_AIRPORT_DFW         0.004295
DESTINATION_AIRPORT_DFW    0.004166
dtype: float64


In [7]:
# Step 8: Model Evaluation
# Evaluating model performance using accuracy and classification report

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Model Accuracy:", accuracy)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Model Accuracy: 0.8088

Classification Report:

              precision    recall  f1-score   support

           0       0.78      0.84      0.80      4705
           1       0.84      0.79      0.81      5295

    accuracy                           0.81     10000
   macro avg       0.81      0.81      0.81     10000
weighted avg       0.81      0.81      0.81     10000



In [8]:
# Step 9: Save Model
import joblib
joblib.dump(model, 'model.pkl')
s3.upload_file('model.pkl', bucket, 'model/model.pkl')